<a href="https://colab.research.google.com/github/zune2/colab-notebooks/blob/main/2026-10-31-bert-tiny-snake-game-epochs3-data5000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# 필수 의존성 패키지인 sentencepiece를 추가하여 설치합니다.
!pip install -q transformers accelerate sentencepiece

import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

# 시드 고정
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"현재 구동 하드웨어 환경: {device}")


현재 구동 하드웨어 환경: cpu


In [6]:
class SnakeEnvironment:
    def __init__(self, width=10, height=10):
        self.width = width
        self.height = height
        self.reset()

    def reset(self):
        # 초기 뱀의 위치 (머리 + 몸통 2칸)
        self.snake = [[5, 5], [5, 6], [5, 7]]
        self.direction = [0, -1] # 초기 방향: 위쪽 (Up)
        self.place_food()
        return self.get_state_inputs()

    def place_food(self):
        while True:
            self.food = [random.randint(0, self.width-1), random.randint(0, self.height-1)]
            if self.food not in self.snake:
                break

    def get_state_inputs(self):
        head = self.snake[0]
        # 현재 진행 방향 기준 왼쪽, 앞쪽, 오른쪽 방향 벡터 계산
        # 현재 방향: [dx, dy] -> 왼쪽: [dy, -dx], 오른쪽: [-dy, dx]
        dx, dy = self.direction
        dir_left = [dy, -dx]
        dir_straight = [dx, dy]
        dir_right = [-dy, dx]

        def is_unsafe(pt):
            if pt[0] < 0 or pt[0] >= self.width or pt[1] < 0 or pt[1] >= self.height:
                return True
            if pt in self.snake[:-1]:
                return True
            return False

        # 주변 장애물 위험 여부 (0: 안전, 1: 위험)
        danger_left = 1 if is_unsafe([head[0] + dir_left[0], head[1] + dir_left[1]]) else 0
        danger_straight = 1 if is_unsafe([head[0] + dir_straight[0], head[1] + dir_straight[1]]) else 0
        danger_right = 1 if is_unsafe([head[0] + dir_right[0], head[1] + dir_right[1]]) else 0

        # 음식의 상대적 위치 (0 또는 1)
        # 진행 방향 기준으로 음식을 향해 회전해야 하는가 판단
        food_left = 1 if (self.food[0] - head[0]) * dir_left[0] + (self.food[1] - head[1]) * dir_left[1] > 0 else 0
        food_straight = 1 if (self.food[0] - head[0]) * dir_straight[0] + (self.food[1] - head[1]) * dir_straight[1] > 0 else 0
        food_right = 1 if (self.food[0] - head[0]) * dir_right[0] + (self.food[1] - head[1]) * dir_right[1] > 0 else 0

        return {
            "danger": [danger_left, danger_straight, danger_right],
            "food": [food_left, food_straight, food_right]
        }

    def step(self, action_idx):
        # action_idx: 0 (좌회전), 1 (직진), 2 (우회전)
        dx, dy = self.direction
        if action_idx == 0:     # 좌회전
            self.direction = [dy, -dx]
        elif action_idx == 2:   # 우회전
            self.direction = [-dy, dx]

        # 머리 이동
        new_head = [self.snake[0][0] + self.direction[0], self.snake[0][1] + self.direction[1]]
        self.snake.insert(0, new_head)

        # 게임 오버 확인
        if (new_head[0] < 0 or new_head[0] >= self.width or
            new_head[1] < 0 or new_head[1] >= self.height or
            new_head in self.snake[1:]):
            return self.get_state_inputs(), -10, True # 사망

        # 음식 섭취 확인
        if new_head == self.food:
            self.place_food()
            return self.get_state_inputs(), 10, False
        else:
            self.snake.pop() # 몸통 유지
            return self.get_state_inputs(), 0, False

# 전문가 의사결정 규칙 (데이터셋 생성용 에이전트)
def get_expert_action(state_info):
    danger = state_info["danger"]
    food = state_info["food"]

    # 1순위: 위험하지 않은 방향 중 음식이 있는 쪽 선택
    for i in range(3):
        if food[i] == 1 and danger[i] == 0:
            return i
    # 2순위: 음식이 없더라도 단순히 위험하지 않은 안전한 방향 선택
    for i in [1, 0, 2]: # 직진 최우선 탐색
        if danger[i] == 0:
            return i
    return 1 # 사방이 막혔을 땐 직진(어차피 사망)


In [7]:
# 상태 객체를 BERT에 주입할 프롬프트형 문자열로 변환
def state_to_sentence(state_info):
    d = state_info["danger"]
    f = state_info["food"]
    return f"danger left {d[0]} straight {d[1]} right {d[2]} food left {f[0]} straight {f[1]} right {f[2]}"

# 대량의 게임 시뮬레이션을 통해 지도학습용 데이터 수집
def generate_dataset(num_samples=5000):
    env = SnakeEnvironment()
    states_text = []
    labels = []

    state_info = env.reset()
    for _ in range(num_samples):
        expert_act = get_expert_action(state_info)

        # 모델의 복원력 향상을 위해 10%의 확률로 노이즈(무작위 행동) 주입
        if random.random() < 0.1:
            action = random.choice([0, 1, 2])
        else:
            action = expert_act

        states_text.append(state_to_sentence(state_info))
        labels.append(action)

        next_state, _, done = env.step(action)
        state_info = next_state if not done else env.reset()

    return states_text, labels

X_text, y_labels = generate_dataset()
print(f"생성된 텍스트 프롬프트 샘플: '{X_text[0]}'")
print(f"해당 상태에 매칭된 행동 인덱스: {y_labels[0]}")


생성된 텍스트 프롬프트 샘플: 'danger left 0 straight 0 right 0 food left 1 straight 1 right 0'
해당 상태에 매칭된 행동 인덱스: 0


In [11]:
# AutoModel 대신 BertModel을 직접 로드합니다.
from transformers import BertTokenizer, BertModel

# 명시적 클래스 호출로 캐시 및 파일 유효성 에러 우회
tokenizer = BertTokenizer.from_pretrained("prajjwal1/bert-tiny")

class SnakeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=32):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

class BertTinySnakePlayer(nn.Module):
    def __init__(self):
        super().__init__()
        # AutoModel 대신 BertModel로 명시적 변경
        self.bert = BertModel.from_pretrained("prajjwal1/bert-tiny")
        # BERT-Tiny의 hidden_size인 128차원을 받아 3가지 행동 스코어로 맵핑
        self.classifier = nn.Linear(128, 3)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # CLS 토큰의 벡터 차원 임베딩 추출
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_output)
        return logits

dataset = SnakeDataset(X_text, y_labels, tokenizer)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

model = BertTinySnakePlayer().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
print("모델 가중치 및 토크나이저 빌드 완료!")


pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 17.8MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


모델 가중치 및 토크나이저 빌드 완료!


In [12]:
model.train()
print("BERT-Tiny 기반 의사결정 레이어 학습 시작...")

for epoch in range(3):
    total_loss = 0
    correct = 0
    for batch in dataloader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / len(dataset) * 100
    print(f"Epoch {epoch+1}/3 - Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

print("학습 완료!")


BERT-Tiny 기반 의사결정 레이어 학습 시작...
Epoch 1/3 - Loss: 0.7812, Accuracy: 69.64%
Epoch 2/3 - Loss: 0.4223, Accuracy: 88.00%
Epoch 3/3 - Loss: 0.3311, Accuracy: 91.48%
학습 완료!


In [13]:
model.eval()
test_env = SnakeEnvironment()
state_info = test_env.reset()
total_score = 0
moves = 0

print("\n--- BERT-Tiny 스네이크 자율 구동 시뮬레이션 시작 ---")

with torch.no_grad():
    for step_idx in range(100): # 최대 100보 이동 검증
        current_sentence = state_to_sentence(state_info)
        inputs = tokenizer(current_sentence, return_tensors="pt", padding='max_length', max_length=32).to(device)

        logits = model(inputs['input_ids'], inputs['attention_mask'])
        ai_action = torch.argmax(logits, dim=1).item()

        next_state, reward, done = test_env.step(ai_action)
        moves += 1

        if reward == 10:
            total_score += 1
            print(f"[{moves}보 돌파] 🍎 음식 섭취 성공! 현재 획득 점수: {total_score}")

        if done:
            print(f"💥 충돌 발생으로 인한 게임 오버. (최종 생존 이동 거리: {moves}보, 최종 점수: {total_score}점)")
            break

        state_info = next_state
    else:
        print(f"🏳️ 최대 이동 제한 도달로 생존 종료. (최종 점수: {total_score}점)")



--- BERT-Tiny 스네이크 자율 구동 시뮬레이션 시작 ---
[8보 돌파] 🍎 음식 섭취 성공! 현재 획득 점수: 1
[17보 돌파] 🍎 음식 섭취 성공! 현재 획득 점수: 2
[21보 돌파] 🍎 음식 섭취 성공! 현재 획득 점수: 3
[24보 돌파] 🍎 음식 섭취 성공! 현재 획득 점수: 4
[29보 돌파] 🍎 음식 섭취 성공! 현재 획득 점수: 5
[31보 돌파] 🍎 음식 섭취 성공! 현재 획득 점수: 6
[38보 돌파] 🍎 음식 섭취 성공! 현재 획득 점수: 7
💥 충돌 발생으로 인한 게임 오버. (최종 생존 이동 거리: 41보, 최종 점수: 7점)


In [15]:
import time
import json
from IPython.display import display, HTML

# 모델 평가 모드 전환
model.eval()

# 시각화용 맵 크기 정의 (10x10 격자)
gui_env = SnakeEnvironment(width=10, height=10)
state_info = gui_env.reset()

total_score = 0
moves = 0
max_steps = 100 # 최대 테스트 이동 횟수

# 1. 최초 1회만 화면에 배치할 HTML 베이스 틀 구조 정의
base_html = """
<div style='text-align: center; font-family: monospace; color: #fff; background: #111; padding: 15px; border-radius: 10px; max-width: 350px; margin: 10px auto;'>
    <h3 style='margin: 0 0 10px 0; color: #ff9f1c;'>🤖 BERT-Tiny 의사결정 GUI</h3>
    <p style='margin: 5px 0;'>🏃 이동 거리: <b id="snake_moves">0보</b> / 100보</p>
    <p style='margin: 5px 0;'>🍎 획득 점수: <b id="snake_score" style='color: #ffbf69;'>0점</b></p>
    <p style='margin: 5px 0; font-size: 11px; color: #888;'>주입된 프롬프트: <br><span id="snake_prompt" style='color: #2ec4b6;'>""</span></p>
</div>

<div id="snake_grid" style='display: grid; grid-template-columns: repeat(10, 30px); gap: 2px; background: #222; padding: 10px; width: 318px; border-radius: 8px; margin: 0 auto;'>
"""

# ID를 부여하여 개별 제어가 가능하도록 100개의 타일 배치
for y in range(10):
    for x in range(10):
        base_html += f'<div id="tile_{x}_{y}" style="width: 30px; height: 30px; background: #333; border-radius: 4px; transition: background 0.05s;"></div>'
base_html += "</div>"

# 화면에 최초 프레임 출력
display(HTML(base_html))

print("🎮 깜빡임 방지 GUI 주행을 시작합니다...")

# 2. 게임 실시간 연산 및 JavaScript 주입 루프
for step_idx in range(max_steps):
    current_sentence = state_to_sentence(state_info)
    inputs = tokenizer(current_sentence, return_tensors="pt", padding='max_length', max_length=32).to(device)

    with torch.no_grad():
        logits = model(inputs['input_ids'], inputs['attention_mask'])
        ai_action = torch.argmax(logits, dim=1).item()

    next_state, reward, done = gui_env.step(ai_action)
    moves += 1
    if reward == 10:
        total_score += 1

    # 현재 맵 상태의 색상 맵을 딕셔너리로 빌드
    color_map = {}
    for y in range(10):
        for x in range(10):
            current_pt = [x, y]
            if gui_env.snake and current_pt == gui_env.snake:
                color = "#2ec4b6" # 머리 (민트색)
            elif current_pt in gui_env.snake:
                color = "#cbf3f0" # 몸통 (연하늘색)
            elif current_pt == gui_env.food:
                color = "#ffbf69" # 음식 (주황색)
            else:
                color = "#333"    # 빈칸
            color_map[f"tile_{x}_{y}"] = color

    # JavaScript 코드를 동적으로 생성하여 돔(DOM) 요소만 직접 교체
    js_update = f"""
    <script>
        (function() {{
            document.getElementById("snake_moves").innerText = "{moves}보";
            document.getElementById("snake_score").innerText = "{total_score}점";
            document.getElementById("snake_prompt").innerText = `{current_sentence}`;

            var colors = {json.dumps(color_map)};
            for (var id in colors) {{
                var el = document.getElementById(id);
                if (el) el.style.background = colors[id];
            }}
        }})();
    </script>
    """
    # clear_output 없이 스크립트만 밀어넣어 내부 갱신
    display(HTML(js_update))

    time.sleep(0.15) # 움직임 속도 조절

    if done:
        print(f"\n💥 충돌 발생으로 AI 구동이 종료되었습니다. (최종 생존: {moves}보, 점수: {total_score}점)")
        break

    state_info = next_state
else:
    print(f"\n🏳️ 최대 이동 제한(100보) 주행 완료! (최종 점수: {total_score}점)")


🎮 깜빡임 방지 GUI 주행을 시작합니다...



💥 충돌 발생으로 AI 구동이 종료되었습니다. (최종 생존: 46보, 점수: 7점)
